## Cell Type Annotation

### Import Required Libraries

This cell imports the necessary Python libraries for automated cell type annotation:

- `celltypist`: A tool for reference-based cell type prediction.
- `models` from `celltypist`: Used to download and manage pretrained annotation models.
- `scanpy`: A widely used package for single-cell RNA-seq data analysis and preprocessing.

These libraries are required to load the data, preprocess it, and perform cell type prediction.

In [20]:
import celltypist
from celltypist import models
import scanpy as sc

### Download and Inspect Available Models

I download all available pretrained CellTypist models (only required once) and print the descriptions of available models to help select the most appropriate one for the dataset.

This ensures that the correct reference model is used for cell type annotation.

In [2]:
# 1. Download the CellTypist models (only need to run this once)
models.download_models(force_update=True)

# 2. View available models to find the most relevant one
print(models.models_description())

📜 Retrieving model list from server https://celltypist.cog.sanger.ac.uk/models/models.json
📚 Total models in list: 61
📂 Storing models in C:\Users\Lenovo\.celltypist\data\models
💾 Downloading model [1/61]: Immune_All_Low.pkl
💾 Downloading model [2/61]: Immune_All_High.pkl
💾 Downloading model [3/61]: Adult_COVID19_PBMC.pkl
💾 Downloading model [4/61]: Adult_CynomolgusMacaque_Hippocampus.pkl
💾 Downloading model [5/61]: Adult_Human_MTG.pkl
💾 Downloading model [6/61]: Adult_Human_PancreaticIslet.pkl
💾 Downloading model [7/61]: Adult_Human_PrefrontalCortex.pkl
💾 Downloading model [8/61]: Adult_Human_Skin.pkl
💾 Downloading model [9/61]: Adult_Human_Vascular.pkl
💾 Downloading model [10/61]: Adult_Mouse_Gut.pkl
💾 Downloading model [11/61]: Adult_Mouse_OlfactoryBulb.pkl
💾 Downloading model [12/61]: Adult_Pig_Hippocampus.pkl
💾 Downloading model [13/61]: Adult_RhesusMacaque_Hippocampus.pkl
💾 Downloading model [14/61]: Adult_cHSPCs_Illumina.pkl
💾 Downloading model [15/61]: Adult_cHSPCs_Ultima.pkl
💾

                                      model  \
0                        Immune_All_Low.pkl   
1                       Immune_All_High.pkl   
2                    Adult_COVID19_PBMC.pkl   
3   Adult_CynomolgusMacaque_Hippocampus.pkl   
4                       Adult_Human_MTG.pkl   
..                                      ...   
56                   Nuclei_Lung_Airway.pkl   
57       PaediatricAdult_COVID19_Airway.pkl   
58         PaediatricAdult_COVID19_PBMC.pkl   
59                      Pan_Fetal_Human.pkl   
60                Thymus_Allograft_PBMC.pkl   

                                          description  
0   immune sub-populations combined from 20 tissue...  
1   immune populations combined from 20 tissues of...  
2   peripheral blood mononuclear cell types from C...  
3   cell types from the hippocampus of adult cynom...  
4   cell types and subtypes (10x-based) from the a...  
..                                                ...  
56  cell populations from snRNA-seq of five

### Preprocess Raw Data and Perform Cell Type Prediction

Cell type annotation using CellTypist is performed with these steps:

1. Loads the raw (unscaled) dataset.
2. Applies normalization (`normalize_total`) to scale counts per cell.
3. Applies log-transformation (`log1p`) to stabilize variance.
4. Runs CellTypist prediction using the selected model (`Immune_All_Low.pkl`) with majority voting enabled.
5. Maps the predicted labels back to the integrated dataset using matching cell indices.

This ensures predictions are made on properly normalized raw data and stored in the main AnnData object.

In [4]:
# 1. Load the original raw file
adata_raw = sc.read_h5ad("combined_raw.h5ad")

# 2. Re-apply the exact preprocessing CellTypist expects
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)

# 3. Run prediction on the unscaled data
predictions = celltypist.annotate(
    adata_raw, 
    model="Adult_Human_PrefrontalCortex.pkl", 
    majority_voting=True
)

# 4. Map predictions back to your integrated/HVG filtered object using the index
adata_raw.obs['celltypist_prediction'] = predictions.predicted_labels.loc[adata_raw.obs_names, 'predicted_labels']
adata_raw.obs['majority_voting'] = predictions.predicted_labels.loc[adata_raw.obs_names, 'majority_voting']

🔬 Input data has 77983 cells and 33538 genes
🔗 Matching reference genes in the model
🧬 3685 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 20
🗳️ Majority voting the predictions
✅ Majority voting done!


In [10]:
adata_raw.obs['majority_voting'].value_counts().to_csv("celltypist_majority_voting_counts.txt")

In [ ]:
'''
# 1. Load the original raw file
adata_raw = sc.read_h5ad("combined_raw.h5ad")

# 2. Re-apply the exact preprocessing CellTypist expects
sc.pp.normalize_total(adata_raw, target_sum=1e4)
sc.pp.log1p(adata_raw)

# 3. Run prediction on the unscaled data
predictions = celltypist.annotate(
    adata_raw, 
    model="Immune_All_Low.pkl", 
    majority_voting=True
)

# 4. Map predictions back to your integrated/HVG filtered object using the index
adata.obs['celltypist_prediction'] = predictions.predicted_labels.loc[adata.obs_names, 'predicted_labels']
adata.obs['majority_voting'] = predictions.predicted_labels.loc[adata.obs_names, 'majority_voting']
'''

🔬 Input data has 77983 cells and 33538 genes
🔗 Matching reference genes in the model
🧬 5967 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 20
🗳️ Majority voting the predictions
✅ Majority voting done!


### Inspect Cell Metadata

It displays the `.obs` dataframe of the AnnData object.

It allows verification that the new CellTypist predictions (`celltypist_prediction` and `majority_voting`) have been successfully added to the metadata.

In [15]:
adata_raw.obs

,sample,condition,batch,celltypist_prediction,majority_voting
AAACCCAAGAATAACC-1-0,GSM6106340_HSDG07HC,control,0,VLMC COL1A2 VLMCA8,VLMC COL1A2 SLC13A3
AAACCCAAGTCATCCA-1-0,GSM6106340_HSDG07HC,control,0,L2-3 CUX2 ACVR1C THSD7A,L2-3 CUX2 ACVR1C THSD7A
AAACCCAGTAATCAGA-1-0,GSM6106340_HSDG07HC,control,0,InN VIP CLSTN2,InN VIP CLSTN2
AAACCCATCCATTCGC-1-0,GSM6106340_HSDG07HC,control,0,L3-5 RORB GABRG1 KCNH7,L3-5 RORB GABRG1 KCNH7
AAACGAAAGGAGTACC-1-0,GSM6106340_HSDG07HC,control,0,L3-5 RORB MKX GRIN3A,L3-5 RORB MKX GRIN3A
...,...,...,...,...,...
TTTGTTGGTTCATCTT-1-11,GSM6106351_hsDG99HC,control,11,Micro P2RY12 APBB1IP,Micro P2RY12 APBB1IP
TTTGTTGTCATCTATC-1-11,GSM6106351_hsDG99HC,control,11,Oligo MOG OPALIN,Oligo MOG OPALIN
TTTGTTGTCCTCGATC-1-11,GSM6106351_hsDG99HC,control,11,L2-3 CUX2 NTNG1 PALMD,L2-3 CUX2 NTNG1 PALMD
TTTGTTGTCGTTAGAC-1-11,GSM6106351_hsDG99HC,control,11,Oligo MOG OPALIN,Oligo MOG OPALIN


In [17]:
#check the shape of the raw data after cell typing
adata_raw.shape

(77983, 33538)

### Export Cell Type Predictions

The predicted cell type labels are exported to a CSV file (`celltypist_results.csv`).

This allows:
- External analysis
- Record keeping
- Sharing results without requiring the AnnData object

In [18]:
predictions.predicted_labels.to_csv("celltypist_results.csv")